# ACTION-ID v0.3.2 — matched handoffs + sequential evidence audits

This notebook runs in Google Colab CPU. **E0 is entirely local. E1 requires your OpenRouter API key; E2 optionally needs a Vercel AI Gateway key.** No synthetic model responses are silently substituted for API calls. Old v0.3.1 handoffs are not scored as new-format results.

## 1. Upload and extract the new experiment ZIP
Upload `ACTION_ID_v0_3_2.zip`. This step must be run once in a fresh Colab runtime.

In [ ]:
from google.colab import files
import zipfile, os
uploaded = files.upload()
assert "ACTION_ID_v0_3_2.zip" in uploaded, "Upload ACTION_ID_v0_3_2.zip"
with zipfile.ZipFile("ACTION_ID_v0_3_2.zip") as z:
    z.extractall("/content")
os.chdir("/content/action_id_v0_3_2")
print("Working directory:", os.getcwd())

## 2. Install and run the 38+ local tests

In [ ]:
!pip -q install -r requirements.txt
!python -m unittest discover -s tests -v

## 3. Offline real-ML baseline, no LLM calls
The 12 fixtures are repeated Iris scenarios; deterministic exact-hint success is **not** an observed agent benefit.

In [ ]:
!python run_offline_v032.py --out results/offline_v032

## 4. OpenRouter settings for E1
Use your own OpenRouter API key, which is entered through a masked prompt. Keep the same model for both handoff formats. Make sure the ID is available on your account.

In [ ]:
from getpass import getpass
os.environ["HANDOFF_CHAT_URL"] = "https://openrouter.ai/api/v1/chat/completions"
os.environ["HANDOFF_MODEL"] = "openai/gpt-oss-120b"
os.environ["HANDOFF_REASONING_EFFORT"] = "low"
os.environ["HANDOFF_API_KEY"] = getpass("OpenRouter API key: ")
print("Configured OpenRouter model:", os.environ["HANDOFF_MODEL"])

## 5. One-case smoke: 2 final handoffs, 6 OpenRouter completions
If either response is unparseable, inspect `final_handoff` and finish reasons before scaling.

In [ ]:
!python run_live_handoffs.py --out results/live_v032_smoke --seeds 21,37 --max-cases 1 --repeats 1 --max-tokens 1600
!python analyze_v032.py --handoffs results/live_v032_smoke/handoffs.jsonl --out results/v032_smoke_analysis.json

## 6. Run the sequential policies on the smoke pair (no paid decision models)

In [ ]:
!python run_sequential.py --handoffs results/live_v032_smoke/handoffs.jsonl --out results/v032_smoke_decisions.jsonl --with-deterministic-bundle
!python analyze_v032.py --handoffs results/live_v032_smoke/handoffs.jsonl --decisions results/v032_smoke_decisions.jsonl --out results/v032_smoke_joint.json

## 7. Optional four-case paired pilot
Cost warning: 4 cases × 2 repetitions × 2 styles × 3 stages = **48 OpenRouter requests**. Do not run before checking the smoke output.

In [ ]:
# Uncomment these commands after reviewing the smoke run:
# !python run_live_handoffs.py --out results/live_v032_pilot --seeds 21,37 --max-cases 4 --repeats 2 --max-tokens 1600
# !python analyze_v032.py --handoffs results/live_v032_pilot/handoffs.jsonl --out results/v032_pilot_analysis.json
# !python run_sequential.py --handoffs results/live_v032_pilot/handoffs.jsonl --out results/v032_pilot_decisions.jsonl --with-deterministic-bundle

## 8. Optional Jev and Liquid d1 on the **same handoffs**
These are *additional decision-model calls through Vercel*, not OpenRouter chat completions. Do not interpret raw decision-model probabilities as evidence authenticity. The snippet runs against the smoke pair; use the pilot path later if appropriate.

In [ ]:
# Uncomment to run Jev/d1 through Vercel AI Gateway:
# os.environ["AI_GATEWAY_API_KEY"] = getpass("Vercel AI Gateway key: ")
# !python run_sequential.py --handoffs results/live_v032_smoke/handoffs.jsonl --out results/v032_smoke_jev_d1.jsonl --policies typesafe-ai/jev,liquid/d1

## 9. Export the experiment results
This ZIP may contain raw provider responses and copied input records; inspect its contents before sharing publicly. API keys are never placed in the output by this notebook.

In [ ]:
import shutil
archive=shutil.make_archive("/content/ACTION_ID_v032_results", "zip", root_dir="/content/action_id_v0_3_2", base_dir="results")
print("Archive:", archive)
files.download(archive)